# PhishTriage: Colab notebook

Canonical development notebook for an academic phishing-email triage project. It runs top to bottom in
Google Colab on **CPU only**. There is no local backend, Docker or database server.

**Implemented phase:** data acquisition, auditing, and leakage-resistant train/dev/test partitions.
Model training starts in the next phase. **Nothing in this notebook fits a model or any learned preprocessing.**

Run with *Runtime → Run all*. Every cell is idempotent, so re-running reuses verified downloads.

## 0. Environment

Installs only packages that are missing. Colab already ships all of them, so normally nothing is installed.
The cell then fixes the random seed and chooses a working directory:

* **Colab:** `/content/phishtriage`, which is temporary and discarded with the runtime.
* **Local:** `$PHISHTRIAGE_WORKDIR` if set, otherwise `~/.cache/phishtriage`, which is outside the Git repository.

Raw data is written only under this working directory and never into the repository.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "matplotlib": "matplotlib", "requests": "requests"}  # import name -> pip name
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import hashlib, json, os, platform, random, re, shutil, time, zlib
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:
    WORKDIR = Path("/content/phishtriage")
else:
    WORKDIR = Path(os.environ.get("PHISHTRIAGE_WORKDIR", Path.home() / ".cache" / "phishtriage"))
RAW_DIR = WORKDIR / "raw"            # verified downloads
MANUAL_DIR = WORKDIR / "manual"      # user-uploaded fallback files
OUT_DIR = WORKDIR / "outputs"        # small aggregate results
FIG_DIR = OUT_DIR / "figures"
for d in (RAW_DIR, MANUAL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Optional: also copy the small aggregate outputs here (used locally to refresh the repo's reports/).
EXPORT_DIR = Path(os.environ["PHISHTRIAGE_EXPORT_DIR"]) if os.environ.get("PHISHTRIAGE_EXPORT_DIR") else None

def display_path(p: Path) -> str:
    """Show paths without exposing the local home directory."""
    try:
        return "~/" + str(Path(p).relative_to(Path.home()))
    except ValueError:
        return str(p)

print(f"Python {platform.python_version()} | Colab: {IN_COLAB} | CPU-only execution")
print(" | ".join(f"{m.__name__} {m.__version__}" for m in (np, pd, scipy, matplotlib, requests)))
print("Working directory:", display_path(WORKDIR))
print("Seed:", SEED)

## 1. Dataset choice and provenance

**First candidate, rejected: MeAJOR v2.0** ([zenodo.org/records/18471483](https://zenodo.org/records/18471483), CC-BY-4.0).
The record says it merges TREC-05/06/07, the Nazario phishing corpus and Nigerian fraud. When inspected on
2026-10-15, its only data file (`meajor_cleaned_preprocessed.parquet.gzip`, 108,685 rows) contained just the sources
`trec5`, `trec6` and `trec7`. The TREC corpora are **spam/ham** collections, so its `label=1` means *spam*, not
phishing. Using it would conflate generic spam with phishing.

**Selected:** *Phishing Email Curated Datasets* by A. I. Champa
([zenodo.org/records/8339691](https://zenodo.org/records/8339691), DOI 10.5281/zenodo.8339691, CC-BY-4.0,
open access, no credentials needed). Each source corpus is a separate CSV file, so the labels can be traced:

| Source file | Original label | Mapped label | Rationale |
|---|---|---|---|
| `Nazario.csv` | 1 (all rows) | **phishing** | Jose Nazario's phishing corpus, collected from a honeypot mailbox |
| `SpamAssasin.csv` | 0 | **legitimate** | SpamAssassin public corpus *ham* |
| `SpamAssasin.csv` | 1 | *excluded* | Generic spam, not phishing |

Known limitations, which are measured below: (a) source and label are perfectly confounded; (b) phishing mail
is dated 2015–2022 while legitimate mail is mostly 2002; (c) the curated `urls` column is a 0/1 flag, and link
targets were removed from the bodies.

In [ ]:
ZENODO_RECORD = "https://zenodo.org/records/8339691"
ZENODO_DOI = "10.5281/zenodo.8339691"
DATASET_LICENSE = "CC-BY-4.0"
CITATION = ("A. I. Champa, M. F. Rabbi, M. F. Zibran, 'Curated Datasets and Feature Analysis for Phishing Email "
            "Detection with Machine Learning', IEEE ICMI 2024.")

# Pinned from the Zenodo API and a verified download on 2026-10-15.
SOURCES = {
    "nazario": {
        "file": "Nazario.csv",
        "url": "https://zenodo.org/api/records/8339691/files/Nazario.csv/content",
        "size": 7_811_841,
        "md5": "4022d055bb7cf8602f30f768f652e91e",
        "sha256": "b8fbc4158fbdfaff1ed98584c43d72e283c6352d7ade4d457d34c1d79488d184",
    },
    "spamassassin": {
        "file": "SpamAssasin.csv",
        "url": "https://zenodo.org/api/records/8339691/files/SpamAssasin.csv/content",
        "size": 14_879_195,
        "md5": "d921e0b2333bfaa058bd38193e6b3fbd",
        "sha256": "3bfe8f8abff89f69a98456be50413c2fcb20a476141116dd84ee1507b980c00e",
    },
}
# (source, original label) -> mapped label; None means excluded, with the reason in EXCLUSION_REASON.
LABEL_MAP = {("nazario", 1): "phishing", ("spamassassin", 0): "legitimate", ("spamassassin", 1): None}
EXCLUSION_REASON = {("spamassassin", 1): "spam_not_phishing"}
REQUIRED_COLUMNS = {"subject", "body", "label"}
OPTIONAL_COLUMNS = {"sender", "receiver", "date", "urls"}
MAX_DOWNLOAD_BYTES = 50_000_000
ALLOW_UPLOAD_FALLBACK = True

## 2. Download with verification

`acquire()` reuses an existing file only if its SHA-256 matches. Otherwise it downloads with timeouts and three
bounded retries, rejects oversized or truncated responses, and checks the SHA-256 before the file is kept.

If the download fails, it falls back to a manual upload: Colab shows a file picker, and locally you place the file in
`manual/`. Uploaded files go through the same schema validation, and a checksum mismatch is recorded, never hidden.
Nothing in this flow continues silently after a failure.

In [ ]:
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def download_verified(name: str, spec: dict, retries: int = 3, timeout=(10, 60)) -> tuple[Path, dict]:
    dest = RAW_DIR / spec["file"]
    if dest.exists():
        if sha256_of(dest) == spec["sha256"]:
            return dest, {"status": "reused_verified", "attempts": 0}
        print(f"{dest.name}: existing file failed the checksum; re-downloading.")
        dest.unlink()
    tmp = dest.with_name(dest.name + ".part")
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            with requests.get(spec["url"], stream=True, timeout=timeout,
                              headers={"Accept-Encoding": "identity"}) as r:
                r.raise_for_status()
                declared = int(r.headers.get("Content-Length", -1))
                if declared > MAX_DOWNLOAD_BYTES:
                    raise ValueError(f"server declares {declared:,} bytes (> {MAX_DOWNLOAD_BYTES:,} limit)")
                if declared not in (-1, spec["size"]):
                    raise ValueError(f"server declares {declared:,} bytes, expected {spec['size']:,}")
                h, n = hashlib.sha256(), 0
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        n += len(chunk)
                        if n > MAX_DOWNLOAD_BYTES:
                            raise ValueError("download exceeded the size limit")
                        f.write(chunk)
                        h.update(chunk)
            if n != spec["size"]:
                raise ValueError(f"received {n:,} bytes, expected {spec['size']:,}")
            if h.hexdigest() != spec["sha256"]:
                raise ValueError("SHA-256 mismatch")
            tmp.replace(dest)
            return dest, {"status": "downloaded_verified", "attempts": attempt}
        except (requests.RequestException, ValueError) as e:
            last_error = e
            tmp.unlink(missing_ok=True)
            print(f"{name}: attempt {attempt}/{retries} failed: {e}")
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status is not None and 400 <= status < 500 and status != 429:
                break  # client errors (e.g. 404) will not fix themselves
            if attempt < retries:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"{name}: download failed after {attempt} attempt(s): {last_error}")


def request_upload(spec: dict) -> Path:
    dest = MANUAL_DIR / spec["file"]
    if IN_COLAB:
        from google.colab import files
        print(f"Please upload {spec['file']} (download it from {ZENODO_RECORD}).")
        uploaded = files.upload()
        if spec["file"] not in uploaded:
            raise RuntimeError(f"Expected an upload named {spec['file']}, got {sorted(uploaded)}")
        dest.write_bytes(uploaded[spec["file"]])
    elif not dest.exists():
        raise RuntimeError(f"Download failed. Place {spec['file']} from {ZENODO_RECORD} at "
                           f"{display_path(dest)} and re-run this cell.")
    return dest


def acquire(name: str, spec: dict) -> tuple[Path, dict]:
    try:
        return download_verified(name, spec)
    except RuntimeError as e:
        if not ALLOW_UPLOAD_FALLBACK:
            raise
        print(f"DOWNLOAD FAILED, falling back to manual upload: {e}")
        path = request_upload(spec)
        ok = sha256_of(path) == spec["sha256"]
        return path, {"status": "uploaded_verified" if ok else "uploaded_CHECKSUM_MISMATCH", "attempts": None}


def load_and_validate(name: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{name}: missing required columns {sorted(missing)}; found {list(df.columns)}")
    if len(df) == 0:
        raise ValueError(f"{name}: file has no rows")
    if df["label"].isna().any():
        raise ValueError(f"{name}: {int(df['label'].isna().sum())} rows have no label")
    unknown = {(name, int(v)) for v in df["label"].unique()} - set(LABEL_MAP)
    if unknown:
        raise ValueError(f"{name}: unmapped (source, label) values {sorted(unknown)}")
    return df


raw, provenance = {}, {}
for name, spec in SOURCES.items():
    path, info = acquire(name, spec)
    df = load_and_validate(name, path)
    raw[name] = df
    provenance[name] = {
        **{k: spec[k] for k in ("file", "url", "size", "md5")},
        "expected_sha256": spec["sha256"],
        "observed_sha256": sha256_of(path),
        **info,
        "rows": len(df),
        "columns": list(df.columns),
        "checked_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    print(f"{name:13s} {info['status']:28s} rows={len(df):,} columns={list(df.columns)}")

if any(p["observed_sha256"] != p["expected_sha256"] for p in provenance.values()):
    print("WARNING: at least one file does not match the pinned checksum. Its results are not comparable to the "
          "documented dataset version.")

## 3. Normalization

The notebook builds one documented dataframe. Only fields that exist are kept. Missing headers, senders, dates and
URLs are **flagged, never invented**.

| Column | Meaning |
|---|---|
| `record_id` | `<source>:<row index in the pinned file>`, a non-sensitive identifier |
| `source` | `nazario` or `spamassassin` (audit only, **not** a model feature) |
| `subject`, `body` | original text, with missing values as empty strings |
| `text` | `subject + blank line + body`, the **only** model input |
| `original_label` | label as given in the source file |
| `mapped_label` | `phishing` / `legitimate`, or empty if excluded |
| `exclusion_reason` | why a row is excluded (empty if included) |
| `missing_*` | whether the source lacks or blanks that field |
| `src_url_flag` | the curator's 0/1 "contains URLs" flag; the URLs themselves are not in the data |
| `year` | year parsed from the `date` header (audit only) |

In [ ]:
def is_missing(s: pd.Series) -> pd.Series:
    return s.isna() | s.fillna("").astype(str).str.strip().eq("")


frames = []
for name, df in raw.items():
    d = pd.DataFrame({
        "record_id": [f"{name}:{i:05d}" for i in range(len(df))],
        "source": name,
        "subject": df["subject"].fillna("").astype(str),
        "body": df["body"].fillna("").astype(str),
        "original_label": df["label"].astype(int),
    })
    for col in ("sender", "receiver", "date", "subject", "body"):
        d[f"missing_{col}"] = is_missing(df[col]) if col in df else True
    d["src_url_flag"] = df["urls"].astype("Int64") if "urls" in df else pd.NA
    d["year"] = (pd.to_datetime(df["date"], errors="coerce", utc=True, format="mixed").dt.year.astype("Int64")
                 if "date" in df else pd.NA)
    frames.append(d)

data = pd.concat(frames, ignore_index=True)
data["text"] = (data["subject"].str.strip() + "\n\n" + data["body"].str.strip()).str.strip()
keys = list(zip(data["source"], data["original_label"]))
data["mapped_label"] = [LABEL_MAP[k] for k in keys]
data["exclusion_reason"] = [EXCLUSION_REASON.get(k, "") for k in keys]
data.loc[data["exclusion_reason"].eq("") & data["text"].eq(""), "exclusion_reason"] = "empty_text"

assert data["record_id"].is_unique
print(f"Loaded rows: {len(data):,}")
print("Normalized columns:", list(data.columns))

## 4. Audit

Every number below is computed from the loaded files.

Duplicates are found on *normalized* text: lower-cased, with whitespace collapsed. When identical texts carry
different labels, **all** copies are excluded as ambiguous. Among the remaining exact duplicates, one copy is kept
(the lowest `record_id`) and the extra copies are excluded.

In [ ]:
def normalize_for_dup(t: str) -> str:
    return re.sub(r"\s+", " ", t.lower()).strip()


data["text_hash"] = [hashlib.sha1(normalize_for_dup(t).encode("utf-8")).hexdigest() for t in data["text"]]

audit = {"rows_loaded": int(len(data)), "rows_per_source": data["source"].value_counts().sort_index().to_dict()}
audit["original_label_by_source"] = {
    s: {str(k): int(v) for k, v in g.value_counts().sort_index().items()}
    for s, g in data.groupby("source")["original_label"]}
audit["missing_fields_by_source"] = {
    s: {c.removeprefix("missing_"): int(g[c].sum()) for c in data.columns if c.startswith("missing_")}
    for s, g in data.groupby("source")}
audit["empty_combined_text"] = int(data["text"].eq("").sum())

# Exact duplicates over all loaded rows (including excluded spam)
dup_all = data.duplicated("text_hash", keep=False)
audit["exact_duplicates_all_rows"] = {
    "rows_in_duplicate_groups": int(dup_all.sum()),
    "duplicate_groups": int(data.loc[dup_all, "text_hash"].nunique()),
    "redundant_copies": int(data.duplicated("text_hash").sum())}

# Conflicting labels: identical text appearing with more than one (source, original label) pair,
# e.g. SpamAssassin ham vs spam, or Nazario phishing vs SpamAssassin ham. All copies become ambiguous.
src_label = data["source"] + ":" + data["original_label"].astype(str)
conflict_hashes = set(src_label.groupby(data["text_hash"]).nunique().loc[lambda s: s > 1].index)
audit["conflicting_duplicate_groups_all_rows"] = len(conflict_hashes)

cand = data["exclusion_reason"].eq("")
data.loc[cand & data["text_hash"].isin(conflict_hashes), "exclusion_reason"] = "conflicting_duplicate_labels"
cand = data["exclusion_reason"].eq("")
extra = data[cand].sort_values("record_id").duplicated("text_hash").reindex(data.index, fill_value=False)
data.loc[extra, "exclusion_reason"] = "exact_duplicate_extra_copy"

data["included"] = data["exclusion_reason"].eq("")
data["label"] = data["mapped_label"].map({"legitimate": 0, "phishing": 1}).where(data["included"]).astype("Int64")
audit["exclusions"] = data.loc[~data["included"], "exclusion_reason"].value_counts().to_dict()
audit["included_by_class"] = data.loc[data["included"], "mapped_label"].value_counts().to_dict()
audit["source_by_class_included"] = pd.crosstab(
    data.loc[data["included"], "source"], data.loc[data["included"], "mapped_label"]).to_dict()

print(json.dumps({k: audit[k] for k in audit if k != "missing_fields_by_source"}, indent=2))
print("\nMissing or blank fields per source:")
display(pd.DataFrame(audit["missing_fields_by_source"]).T)

### Confounds a classifier could exploit

Each source has a different collection context, so some text patterns identify the **source** rather than phishing
behaviour. This cell measures a fixed list of such artifacts per class, plus the date ranges.
The next phase will use these numbers to decide on masking and on stress tests.

In [ ]:
ARTIFACTS = {
    "collector mailbox (monkey.org)": r"monkey\.org",
    "collector name ('jose')": r"\bjose\b",
    "mailing-list footer": r"listinfo|unsubscribe",
    "SpamAssassin / SourceForge mention": r"spamassassin|sourceforge",
    "bracketed list tag in subject": r"^\s*(?:re:\s*)?\[[^\]]+\]",
    "literal http(s) link": r"https?://",
}
inc = data[data["included"]]
lower = inc["text"].str.lower()
artifact_rates = pd.DataFrame({
    label: {k: round(float(lower[inc["mapped_label"] == label].str.contains(p, regex=True).mean()), 3)
            for k, p in ARTIFACTS.items()}
    for label in ("legitimate", "phishing")})
audit["artifact_prevalence_by_class"] = artifact_rates.to_dict()

years = inc.groupby("mapped_label")["year"].describe(percentiles=[0.05, 0.5, 0.95])[["count", "min", "5%", "50%", "95%", "max"]]
audit["year_by_class"] = {k: {c: (None if pd.isna(v) else float(v)) for c, v in row.items()} for k, row in years.iterrows()}
audit["src_url_flag_rate_by_class"] = inc.groupby("mapped_label")["src_url_flag"].mean().round(3).to_dict()

print("Share of included texts containing each artifact:")
display(artifact_rates)
print("Year parsed from the date header:")
display(years)

### Sanitized examples

Short excerpts with email addresses, URLs, IP addresses and phone-like numbers redacted. Only a few are shown,
for orientation.

In [ ]:
REDACTIONS = [
    (re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"), "[EMAIL]"),
    (re.compile(r"(?:https?://|www\.)\S+", re.I), "[URL]"),
    (re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b"), "[IP]"),
    (re.compile(r"\+?\d[\d ().-]{7,}\d"), "[NUMBER]"),
    # Names in reply attributions ("Jane Doe wrote:", '"J" == Jane writes:') and forwarded headers.
    (re.compile(r"(?:\"\w+\"\s*==\s*)?(?:[A-Z][\w.'-]*\s+){0,3}[A-Z][\w.'-]*\s+(?=(?:wrote|writes)\b)"), "[NAME] "),
    (re.compile(r"\b(From|To|Cc):\s*.*?(?=\s+(?:From|To|Cc|Sent|Date|Subject):|$)"), r"\1: [REDACTED]"),
]


def sanitize(t: str, limit: int = 160) -> str:
    t = re.sub(r"\s+", " ", t).strip()
    for pattern, token in REDACTIONS:
        t = pattern.sub(token, t)
    return t[:limit] + ("…" if len(t) > limit else "")


examples = pd.concat([g.sample(2, random_state=SEED) for _, g in inc.groupby("mapped_label")])
examples = examples[["record_id", "mapped_label", "subject", "body"]].copy()
examples["subject"] = examples["subject"].map(lambda s: sanitize(s, 70))
examples["body"] = examples["body"].map(sanitize)
with pd.option_context("display.max_colwidth", 170):
    display(examples.reset_index(drop=True))

### Class balance and text length

In [ ]:
# Palette slots 1 and 2 (blue = legitimate, orange = phishing); recessive axes, direct labels.
CLASS_COLORS = {"legitimate": "#2a78d6", "phishing": "#eb6834"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
                     "axes.titlecolor": INK, "axes.titlesize": 11, "font.size": 9})

counts = inc["mapped_label"].value_counts().reindex(["legitimate", "phishing"])
excl = data.loc[~data["included"], "exclusion_reason"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1, 1.3]})
ax = axes[0]
ax.barh(counts.index, counts.values, color=[CLASS_COLORS[c] for c in counts.index], height=0.5)
for i, v in enumerate(counts.values):
    ax.text(v, i, f" {v:,}", va="center", color=INK)
ax.set_title("Included records by class")
ax.set_xlabel("records"); ax.invert_yaxis(); ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.set_xlim(0, counts.max() * 1.2)

ax = axes[1]
ax.barh(excl.index, excl.values, color=INK_2, height=0.5)
for i, v in enumerate(excl.values):
    ax.text(v, i, f" {v:,}", va="center", color=INK)
ax.set_title("Excluded records by reason")
ax.set_xlabel("records"); ax.invert_yaxis(); ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.set_xlim(0, excl.max() * 1.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_balance.png", dpi=120)
plt.show()

lengths = inc.assign(chars=inc["text"].str.len())
bins = np.logspace(1, np.log10(max(lengths["chars"].max(), 100)), 50)
fig, ax = plt.subplots(figsize=(8, 3.2))
for label in ("legitimate", "phishing"):
    vals = lengths.loc[lengths["mapped_label"] == label, "chars"]
    ax.hist(vals, bins=bins, histtype="step", lw=2, color=CLASS_COLORS[label],
            label=f"{label} (median {int(vals.median()):,} chars)")
ax.set_xscale("log"); ax.set_xlabel("combined subject + body length (characters, log scale)"); ax.set_ylabel("records")
ax.set_title("Text length by class"); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK)
fig.tight_layout()
fig.savefig(FIG_DIR / "text_length.png", dpi=120)
plt.show()

audit["text_length_chars_by_class"] = {
    k: {s: float(v) for s, v in g.describe(percentiles=[0.05, 0.5, 0.95])[["min", "5%", "50%", "95%", "max"]].items()}
    for k, g in lengths.groupby("mapped_label")["chars"]}

## 5. Leakage-resistant partitions

**Duplicate grouping.** Exact duplicates were already collapsed. Near-duplicates are found by **exact Jaccard
similarity** over word 5-gram shingle sets: lower-cased alphanumeric tokens, with each shingle hashed using CRC32,
which is deterministic. Pairs with Jaccard ≥ 0.8 are linked, and connected components become `group_id`s. A text
shorter than five tokens forms a single shingle. This catches template variants that differ in a few words. It does
not catch paraphrases or heavy re-wording.

**Split.** Whole groups are assigned to train/dev/test (about 70/15/15), stratified by the group's majority label
with a seeded shuffle. Nothing is fitted on the data here.

**Model input.** `text` only. `source`, labels, IDs, dates and the curator's URL flag are never features.

In [ ]:
NEAR_DUP_JACCARD = 0.8
SHINGLE_WORDS = 5
SPLIT_FRACTIONS = {"train": 0.70, "dev": 0.15, "test": 0.15}
MIN_PER_CLASS_PER_SPLIT = 50
MODEL_INPUT_COLUMNS = ["text"]
NON_FEATURE_COLUMNS = {"record_id", "source", "original_label", "mapped_label", "label", "exclusion_reason",
                       "included", "group_id", "split", "year", "src_url_flag", "text_hash"}


def shingle_ids(t: str) -> set[int]:
    toks = re.findall(r"[a-z0-9]+", t.lower())
    if len(toks) < SHINGLE_WORDS:
        return {zlib.crc32(" ".join(toks).encode())}
    return {zlib.crc32(" ".join(toks[i:i + SHINGLE_WORDS]).encode()) for i in range(len(toks) - SHINGLE_WORDS + 1)}


def near_duplicate_groups(texts: list[str], threshold: float, block: int = 500) -> tuple[np.ndarray, int]:
    sets = [shingle_ids(t) for t in texts]
    vocab = {h: j for j, h in enumerate(sorted(set().union(*sets)))}
    indptr = np.cumsum([0] + [len(s) for s in sets])
    indices = np.fromiter((vocab[h] for s in sets for h in sorted(s)), dtype=np.int64, count=indptr[-1])
    X = csr_matrix((np.ones(len(indices), dtype=np.int32), indices, indptr), shape=(len(sets), len(vocab)))
    sizes = np.asarray(X.sum(axis=1)).ravel()
    rows, cols = [], []
    for start in range(0, X.shape[0], block):
        inter = (X[start:start + block] @ X.T).tocoo()
        i, j, v = inter.row + start, inter.col, inter.data
        keep = i < j
        i, j, v = i[keep], j[keep], v[keep]
        jac = v / (sizes[i] + sizes[j] - v)
        hit = jac >= threshold
        rows.append(i[hit]); cols.append(j[hit])
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    graph = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(sets)))
    n_comp, comp = connected_components(graph, directed=False)
    return comp, len(rows)


model_df = data[data["included"]].sort_values("record_id").reset_index(drop=True)
t0 = time.time()
comp, n_pairs = near_duplicate_groups(model_df["text"].tolist(), NEAR_DUP_JACCARD)
model_df["group_id"] = [f"g{c:05d}" for c in comp]
group_sizes = model_df["group_id"].value_counts()
mixed = model_df.groupby("group_id")["label"].nunique().gt(1)
audit["near_duplicates"] = {
    "method": f"exact Jaccard on word {SHINGLE_WORDS}-gram shingles (CRC32), threshold {NEAR_DUP_JACCARD}",
    "linked_pairs": int(n_pairs),
    "groups_total": int(group_sizes.size),
    "multi_record_groups": int((group_sizes > 1).sum()),
    "records_in_multi_record_groups": int(group_sizes[group_sizes > 1].sum()),
    "largest_group": int(group_sizes.max()),
    "mixed_label_groups": int(mixed.sum()),
}
print(f"Near-duplicate grouping took {time.time() - t0:.1f}s")
print(json.dumps(audit["near_duplicates"], indent=2))

In [ ]:
def grouped_stratified_split(df: pd.DataFrame, fractions: dict, seed: int) -> pd.Series:
    rng = np.random.default_rng(seed)
    groups = df.groupby("group_id").agg(n=("record_id", "size"), y=("label", lambda s: int(s.mean() >= 0.5)))
    names = list(fractions)
    edges = np.cumsum([fractions[k] for k in names])
    assignment = {}
    for _, g in groups.groupby("y"):
        order = np.array(sorted(g.index))
        rng.shuffle(order)
        cum = np.cumsum(g.loc[order, "n"].to_numpy()) / g["n"].sum()
        idx = np.minimum(np.searchsorted(edges, cum, side="left"), len(names) - 1)
        assignment.update(zip(order, (names[i] for i in idx)))
    return df["group_id"].map(assignment)


model_df["split"] = grouped_stratified_split(model_df, SPLIT_FRACTIONS, SEED)

# ---- leakage and coverage assertions ----
assert model_df["split"].notna().all(), "every record must be assigned"
assert model_df["record_id"].is_unique, "record overlap"
split_of_group = model_df.groupby("group_id")["split"].nunique()
assert (split_of_group == 1).all(), "a duplicate group spans more than one split"
split_sets = {s: set(g) for s, g in model_df.groupby("split")["group_id"]}
for a in SPLIT_FRACTIONS:
    for b in SPLIT_FRACTIONS:
        if a < b:
            assert not (split_sets[a] & split_sets[b]), f"group overlap {a}/{b}"
assert model_df.groupby("text_hash")["split"].nunique().max() == 1, "identical text in more than one split"
per_class = pd.crosstab(model_df["split"], model_df["mapped_label"]).reindex(list(SPLIT_FRACTIONS))
assert (per_class >= MIN_PER_CLASS_PER_SPLIT).all().all(), f"class coverage too low:\n{per_class}"
again = grouped_stratified_split(model_df, SPLIT_FRACTIONS, SEED)
assert again.equals(model_df["split"]), "split is not reproducible"
assert not set(MODEL_INPUT_COLUMNS) & NON_FEATURE_COLUMNS, "source/label/ID columns must not be model inputs"
shares = model_df["split"].value_counts(normalize=True)
for s, f in SPLIT_FRACTIONS.items():
    assert abs(shares[s] - f) < 0.03, f"{s} share {shares[s]:.3f} is far from {f}"
print("All partition assertions passed.")

audit["splits"] = {s: {"records": int((model_df["split"] == s).sum()),
                       "share": round(float(shares[s]), 4),
                       "groups": len(split_sets[s]),
                       **{k: int(v) for k, v in per_class.loc[s].items()}} for s in SPLIT_FRACTIONS}
display(per_class.assign(total=per_class.sum(axis=1), share=shares.reindex(per_class.index).round(3)))

## 6. Save small, safe evidence

These are written to `outputs/` in the working directory:

* `dataset_provenance.json`: source URLs, checksums, retrieval status and label mapping.
* `audit_summary.json`: aggregate counts only, with no message text.
* `split_manifest.csv`: `record_id, group_id, split, mapped_label`. It contains no text, addresses or hashes of
  text, only row indices into the pinned public files.
* `figures/*.png`

Raw data stays in `raw/` and is never exported.

In [ ]:
provenance_doc = {
    "dataset": "Phishing Email Curated Datasets (Champa)",
    "record": ZENODO_RECORD, "doi": ZENODO_DOI, "license": DATASET_LICENSE, "citation": CITATION,
    "files": provenance,
    "label_mapping": [{"source": s, "original_label": l, "mapped_label": m,
                       "exclusion_reason": EXCLUSION_REASON.get((s, l), "")} for (s, l), m in LABEL_MAP.items()],
    "rejected_candidates": [{
        "record": "https://zenodo.org/records/18471483", "name": "MeAJOR v2.0",
        "reason": "Data file contains only TREC-05/06/07 sources (spam/ham); label 1 is spam, not phishing."}],
    "seed": SEED,
    "environment": "colab" if IN_COLAB else "local",
    "versions": {m.__name__: m.__version__ for m in (np, pd, scipy, matplotlib, requests)} | {"python": platform.python_version()},
}
(OUT_DIR / "dataset_provenance.json").write_text(json.dumps(provenance_doc, indent=2) + "\n")
(OUT_DIR / "audit_summary.json").write_text(json.dumps(audit, indent=2, default=int) + "\n")
model_df[["record_id", "group_id", "split", "mapped_label"]].to_csv(OUT_DIR / "split_manifest.csv", index=False)

if EXPORT_DIR:
    (EXPORT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    for p in ["dataset_provenance.json", "audit_summary.json", "split_manifest.csv",
              "figures/class_balance.png", "figures/text_length.png"]:
        shutil.copy2(OUT_DIR / p, EXPORT_DIR / p)
    print("Exported evidence to", display_path(EXPORT_DIR))

for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT_DIR)}  {p.stat().st_size:,} bytes")

In [ ]:
print("PHASE 1 COMPLETE: data acquired, audited and partitioned.")
print(f"  files: " + ", ".join(f"{k}={v['status']}" for k, v in provenance.items()))
print(f"  loaded={audit['rows_loaded']:,}  included={int(data['included'].sum()):,}  excluded={int((~data['included']).sum()):,}")
print("  " + "  ".join(f"{s}={v['records']:,} (legit {v['legitimate']:,} / phish {v['phishing']:,})"
                       for s, v in audit["splits"].items()))